In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../../../archive/final_dataset_with_all_features.csv')

selected_columns = [
    'url_len', 'digits', 'letters', 'domain_ngram_entropy',
    'path_depth', 'path_entropy',
    'consonant_ratio', 'vowel_ratio', 'digit_ratio', 'avg_token_length',
    'token_count', 'label'
]

df = df[selected_columns]

X = df.drop('label', axis=1)
y = df['label']

print(f"\nDataset shape: {df.shape}")
print(f"Class distribution:\n{y.value_counts()}")
print(f"\nFeature names: {X.columns.tolist()}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)



Dataset shape: (651000, 12)
Class distribution:
label
0    428103
1     96457
2     93920
3     32520
Name: count, dtype: int64

Feature names: ['url_len', 'digits', 'letters', 'domain_ngram_entropy', 'path_depth', 'path_entropy', 'consonant_ratio', 'vowel_ratio', 'digit_ratio', 'avg_token_length', 'token_count']


In [5]:
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'Ada Boost': AdaBoostClassifier(n_estimators=100, random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')

    results[name] = model
    print(f"Cross-validation accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
    print(f"Test accuracy: {(y_pred == y_test).mean():.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

Cross-validation accuracy: 0.9162 (+/- 0.0007)
Test accuracy: 0.9171

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.96      0.95     85621
           1       0.95      0.98      0.97     19291
           2       0.77      0.63      0.70     18784
           3       0.98      0.93      0.96      6504

    accuracy                           0.92    130200
   macro avg       0.91      0.88      0.89    130200
weighted avg       0.91      0.92      0.91    130200

Cross-validation accuracy: 0.8920 (+/- 0.0012)
Test accuracy: 0.8917

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.99      0.94     85621
           1       0.86      0.93      0.90     19291
           2       0.85      0.47      0.60     18784
           3       0.92      0.76      0.83      6504

    accuracy                           0.89    130200
   macro avg       0.88      0.79      0.82    130200

In [6]:
import joblib
for k,v in results.items():
    joblib.dump(v, k + ".pkl")